# Language Models (1): n-gram models
- Preprocessing (save transcription data individual text files)
- Build n-gram language models using NLTK
- Use LMs to generate texts

## Prerequisite: Google Drive Mounted Data Path

The notebook uses a data folder that is linked to one of your folders in Google Drive.

In [ ]:
from google.colab import drive
# Mount google drive
drive.mount('/content/drive')
# Specify the folder path where the datasets reside
data_folder = '/content/drive/MyDrive/AMLH/week8-NLP2/lab/student'

Mounted at /content/drive


In [ ]:
import pandas as pd
import nltk
import nltk.data
from nltk.corpus import PlaintextCorpusReader

## 1. Create an NLTK corpus using MTSamples
We want to use NLTK's built-in functions for *n*-grams and other text processing tasks. To facilitate this, we will create a customised corpus by saving files in a designated folder.

In [ ]:
# Let's load our IHI version of mtsamples
mt_samples_file = f'{data_folder}/mtsamples_ihi.csv'
df = pd.read_csv(mt_samples_file)
df.head()

,description,medical_specialty,sample_name,transcription,keywords
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,SUBJECTIVE:\n\n This 23-year-old white female...,"allergy / immunology, allergic rhinitis, aller..."
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,PAST MEDICAL HISTORY:\n\n He has difficulty cl...,"bariatrics, laparoscopic gastric bypass, weigh..."
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,HISTORY OF PRESENT ILLNESS: \n\n I have seen A...,"bariatrics, laparoscopic gastric bypass, heart..."
3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,2-D M-MODE: \n\n \n\n1. Left atrial enlargeme...,"cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


### 1.1 Save plain text to folder
- Please ensure you create a folder called `mtsamples` where the files will be stored
- You only need to run this block once

In [ ]:
import re
import codecs
from os.path import join
from os import mkdir

def save_string(txt, file_path, encoding='utf-8'):
    with codecs.open(file_path, 'w', encoding=encoding) as wf:
        wf.write(txt)

# Specify where (create the folder first) to store your 5k or so free-text files
text_data_folder = f'./mtsamples'
mkdir(text_data_folder)

print(f'{text_data_folder} created, saving files...')

# Iterate over rows where trascription is not null
for i, r in df[df['transcription'].notnull()].iterrows():
    save_string(r['transcription'], join(text_data_folder, '%s.txt' %(i)))

print('%s files save to %s' % (i + 1, text_data_folder))

./mtsamples created, saving files...
4999 files save to ./mtsamples


### 1.2 Load the texts as a corpus into NLTK

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Create an NLTK corpus using text files
mtsamples = PlaintextCorpusReader(text_data_folder, '.*txt')

# Print second sentence in the corpus (first sentence is a heading - not very representative)
a_sentence = mtsamples.sents()[1]
print(a_sentence)

['This', '23', '-', 'year', '-', 'old', 'white', 'female', 'presents', 'with', 'complaint', 'of', 'allergies', '.']


### 1.3 Let's try some n-grams

In [ ]:
from nltk import bigrams, trigrams, ngrams
from collections import Counter, defaultdict

# Get the bigrams
print(list(bigrams(a_sentence)))

# Get the padded bigrams
print(list(bigrams(a_sentence, pad_left=True, pad_right=True)))

# Get the trigrams
print(list(trigrams(a_sentence)))

# Get the padded trigrams
print(list(trigrams(a_sentence, pad_left=True, pad_right=True)))

[('This', '23'), ('23', '-'), ('-', 'year'), ('year', '-'), ('-', 'old'), ('old', 'white'), ('white', 'female'), ('female', 'presents'), ('presents', 'with'), ('with', 'complaint'), ('complaint', 'of'), ('of', 'allergies'), ('allergies', '.')]
[(None, 'This'), ('This', '23'), ('23', '-'), ('-', 'year'), ('year', '-'), ('-', 'old'), ('old', 'white'), ('white', 'female'), ('female', 'presents'), ('presents', 'with'), ('with', 'complaint'), ('complaint', 'of'), ('of', 'allergies'), ('allergies', '.'), ('.', None)]
[('This', '23', '-'), ('23', '-', 'year'), ('-', 'year', '-'), ('year', '-', 'old'), ('-', 'old', 'white'), ('old', 'white', 'female'), ('white', 'female', 'presents'), ('female', 'presents', 'with'), ('presents', 'with', 'complaint'), ('with', 'complaint', 'of'), ('complaint', 'of', 'allergies'), ('of', 'allergies', '.')]
[(None, None, 'This'), (None, 'This', '23'), ('This', '23', '-'), ('23', '-', 'year'), ('-', 'year', '-'), ('year', '-', 'old'), ('-', 'old', 'white'), ('old'

## 2. Create a trigram language model
- Step 1: Iterate through each trigram in your text data and count how often they appear
- Step 2: Convert these frequency counts into probabilities
- **NB** Python `defaultdict` is quite convinient for frequency counting. Check https://docs.python.org/3/library/collections.html#collections.defaultdict if you are not familiar with it.

### Step 1. Iterate through each trigram to count their frequencies

In [ ]:
# Train a trigram model, this process may take a minute or so
model = defaultdict(lambda: defaultdict(lambda: 0))

for sentence in mtsamples.sents():
    # Pad both sides
    for w1, w2, w3 in trigrams(sentence, pad_right=True, pad_left=True):
        model[(w1, w2)][w3] += 1

### Let's examine the frequency counts of a few trigrams

In [ ]:
print(model['presents', 'with']['complaint']) # "complaint" follows "presents" and "with" 2 times
print(model['presents', 'with']['symptoms']) # how about "symptoms"?

print(model[None, None]["This"]) # "this" starts a sentence for 4,346 times

2
4
4346


### [Task 1] Convert frequency counts into probabilities
- Implement the code to convert the frequency counts in the model into probabilities

If you encounter a `float division by zero error`, it likely means you are attempting to use a trigram that does not exist in the model.

In [ ]:
# Let's transform the counts into probabilities
for w1_w2 in model:
    # Calculate the total frequencies of trigrams that start with [w1, w2]
    total_count = float(sum(model[w1_w2].values()))
    for w3 in model[w1_w2]:
        model[w1_w2][w3] /= total_count

### Let's do a quick check of the probability of our old example

In [ ]:
print(model['presents', 'with']['symptoms']) # probability of "symptoms" following "presents with"

0.02857142857142857


## 3. Use your LM
### 3.1 Generate some texts

In [ ]:
import random


def generate_text():
    text = [None, None]

    sentence_finished = False

    while not sentence_finished:
        # r is the random threshold
        r = random.random()
        accumulator = .0

        for word in model[tuple(text[-2:])].keys():
            # keep accumulating probabilities
            accumulator += model[tuple(text[-2:])][word]
            # when the accumulated prob equals/exceeds the random threshold,
            # add the word to the sentence
            if accumulator >= r:
                text.append(word)
                break
        # generating until nothing could be generated
        if text[-2:] == [None, None]:
            sentence_finished = True
    return ' '.join([t for t in text if t])

for i in range(10):
    print(i, generate_text())

0 Both of these .
1 At this point to proceed .
2 3 . 5 , hip including knee .
3 Mental Status Exam : Normal sensation to the subcutaneous tissue , which we have a screw was proximal to the level of the left atrium , A6 to 9 Hz alpha activity .
4 Bilateral obturator pelvic lymphadenectomy and radical excision and decompression .
5 INDICATION :
6 1 .
7 I am going to go ahead and check hemoglobin A1c and spot urine for microalbumin was 9 . 00 , OU : Far VA 20 / 93 .
8 Care was taken to recovery room in good condition .
9 At the end fragments of bone .


### 3.2 Sentence completion using the LM
### **[Task 2]**
- Implement the `complete_sentence` function below that can complete a given seed *partial* sentence
- Ideally, your function should be able to generate various sensible sentences

In [ ]:
def complete_sentence(m, text, topk=10):

    while True:

        candidates = m[tuple(text[-2:])]

        if not candidates:
            return None

        ranked = sorted(
            candidates.items(),
            key=lambda x: x[1],
            reverse=True
        )

        word = random.choice(
            ranked[:topk]
        )[0]

        text.append(word)

        if text[-2:] == [None, None]:
            break

    return " ".join(
        t for t in text if t
    )


seed_sentence = 'A 67-year-old male with COPD, who presents with a 3-day history of increased'
for k in range(10):
    print(k, complete_sentence(model, seed_sentence.split()))

0 A 67-year-old male with COPD, who presents with a 3-day history of increased signal on T1 and gradient echo T2 series # 8 endotracheal tube anesthesia was given intravenous steroid therapy .
1 A 67-year-old male with COPD, who presents with a 3-day history of increased activity in the supine straight leg raise testing evokes back pain that has since applied for hemostasis in all aspects of this area was injected through his ureter on which was negative in August 2006 and developed cough with right upper thigh .
2 A 67-year-old male with COPD, who presents with a 3-day history of increased lower back while working for him at this stage .
3 A 67-year-old male with COPD, who presents with a 3-day history of increased T2 signal within the past few months .
4 A 67-year-old male with COPD, who presents with a 3-day history of increased insertional activity .
5 A 67-year-old male with COPD, who presents with a 3-day history of increased insertional activity , but no other symptoms including